# 第 1 節｜BERT 遮罩填空與中文情緒分類

**本節只做推論，不訓練模型。** 從現成模型讀取文字、輸出結果。先執行安裝與匯入 Cell，再依序完成 **4 組「看範例 → 自己輸入」** 練習。

- **遮罩填空**：在句子裡放入 `[MASK]`，讓 BERT 猜一個 Token。
- **情緒分類**：輸入一句中文，讓已微調模型判斷情緒。
- 每次只測試**一句文字**，不必學 `for` 迴圈。

建議使用 Google Colab 的 T4 GPU；第一次需要連網下載模型，尤其情緒模型檔案較大。


## 準備環境｜直接執行以下兩格，不需要手動輸入


In [ ]:
# Install the libraries required by this notebook.
# 安裝 Notebook 所需套件。
%pip -q install transformers==4.57.1 datasets==3.6.0 accelerate sentencepiece


In [ ]:
# Import the libraries used in this lesson.
# 匯入本節使用的套件。
import torch
import pandas as pd
from transformers import pipeline

# Use GPU when available, otherwise use CPU.
# 若有 GPU 就使用 GPU，否則使用 CPU。
device = 0 if torch.cuda.is_available() else -1

print("GPU available:", torch.cuda.is_available())


## 一、BERT 遮罩填空

`[MASK]` 是一個 Token 的空格，不一定等於一個中文字。`pipeline()` 負責載入現成模型；第一次載入可能需要等候。


### 練習 1｜載入遮罩填空模型

請先執行此格。模型會下載到 Colab；不用每試一句話就重新載入。

```python
# Load a pretrained BERT model for filling masks.
# 載入已預訓練的 BERT 遮罩填空模型。
mask_model = pipeline("fill-mask", model="google-bert/bert-base-chinese", device=device)
```


In [ ]:
# 請在此處根據範例練習程式碼


### 練習 2｜讓 BERT 猜一個字

只需要修改 `text` 的句子，就能重複嘗試。`top_k=3` 會找出前三個候選答案；這裡只顯示第一個。

```python
# Write one sentence with a mask.
# 輸入一句包含 [MASK] 的句子。
text = "今天天氣真[MASK]。"

# Predict the three most likely answers.
# 預測最可能的三個答案。
result = mask_model(text, top_k=3)

# Show the most likely token and its score.
# 顯示第一個候選文字及分數。
print("預測文字：", result[0]["token_str"])
print("分數：", round(result[0]["score"], 3))
```


In [ ]:
# 請在此處根據範例練習程式碼


## 二、中文情緒分類

接著使用 `Johnson8187/Chinese-Emotion-Small`，這是**已完成情緒微調的 mDeBERTa 模型**，不是上一個原生 BERT 遮罩模型。它可以辨識八類情緒，但可能輸出 `LABEL_0`、`LABEL_1` 等代碼。


### 認識 `pipeline()`：先選任務，再選模型

`pipeline()` 是 Hugging Face Transformers 提供的簡易**模型推論工具**。我們先選擇要做的任務（`task`），再指定能完成該任務的模型（`model`）；不用自己撰寫繁複的文字轉換和推論程式。

**請記住：任務和模型必須相容。** 不是每個 BERT 模型都能直接拿來做情緒分類。例如，預訓練的遮罩填空模型適合 `fill-mask`；要分類文字，應使用已具備文字分類能力的模型並搭配 `text-classification`。

**本節有兩種任務：**

- **遮罩填空：**`pipeline("fill-mask", model="google-bert/bert-base-chinese")`，讓模型猜 `[MASK]` 位置的文字。
- **情緒分類：**`pipeline("text-classification", model="Johnson8187/Chinese-Emotion-Small")`，讓已完成情緒分類訓練的模型判斷一句話的情緒。

**現在要做的是「文字分類」**：所以要選能進行文字分類的模型 `Johnson8187/Chinese-Emotion-Small`，`pipeline` 的任務也要設為 **`text-classification`**。

**Transformers 4.57.1 支援的 `pipeline()` 任務名稱：**

| 類型 | `task` 名稱 | 用途（簡單理解） |
|---|---|---|
| 文字 | `fill-mask` | 預測句子中 `[MASK]` 位置的文字 |
| 文字 | `text-classification` | 將整句文字分類，例如情緒或參加意願 |
| 文字 | `token-classification` | 替文字中的詞語分類，例如辨認人名與地名 |
| 文字 | `zero-shot-classification` | 用自己指定的候選類別分類文字 |
| 文字 | `question-answering` | 根據提供的文章回答問題 |
| 文字 | `summarization` | 將長文章整理成摘要 |
| 文字 | `translation` | 把文字翻譯成另一種語言 |
| 文字 | `translation_xx_to_yy` | 指定來源與目標語言的翻譯名稱格式，例如 `translation_en_to_fr` |
| 文字 | `text2text-generation` | 輸入文字，產生另一段文字 |
| 文字 | `text-generation` | 接續文字或產生新的文字內容 |
| 文字 | `feature-extraction` | 將文字轉成模型使用的數值特徵 |
| 影像／影片 | `image-classification` | 判斷圖片屬於哪個類別 |
| 影像／影片 | `object-detection` | 找出圖片中的物件與位置 |
| 影像／影片 | `image-segmentation` | 把圖片分成不同區域 |
| 影像／影片 | `depth-estimation` | 估計圖片中物體的遠近 |
| 影像／影片 | `image-to-text` | 根據圖片產生文字描述 |
| 影像／影片 | `image-to-image` | 由輸入圖片產生另一張圖片 |
| 影像／影片 | `image-feature-extraction` | 將圖片轉成數值特徵 |
| 影像／影片 | `mask-generation` | 產生圖片的區域遮罩 |
| 影像／影片 | `keypoint-matching` | 找出兩張圖片中對應的特徵點 |
| 影像／影片 | `video-classification` | 辨認影片的類別或活動 |
| 影像／影片 | `zero-shot-image-classification` | 依指定的候選類別分類圖片 |
| 影像／影片 | `zero-shot-object-detection` | 依指定物件名稱尋找圖片中的物件 |
| 聲音 | `audio-classification` | 辨認聲音種類，例如音效類別 |
| 聲音 | `automatic-speech-recognition` | 將語音轉成文字 |
| 聲音 | `text-to-audio` | 把文字轉成語音或聲音 |
| 聲音 | `zero-shot-audio-classification` | 依指定的候選類別辨認聲音 |
| 跨模態／文件 | `image-text-to-text` | 結合圖片與文字輸入，產生文字回答 |
| 跨模態／文件 | `visual-question-answering` | 根據圖片回答問題 |
| 跨模態／文件 | `document-question-answering` | 閱讀文件圖片並回答問題 |
| 跨模態／文件 | `table-question-answering` | 根據資料表回答問題 |

**補充：**`sentiment-analysis` 是 `text-classification` 的別名；`ner` 是 `token-classification` 的別名；`text-to-speech` 是 `text-to-audio` 的別名。不同任務需要相容的模型，**不能只改 `task` 名稱，就期待同一個模型完成所有工作**。部分任務還需要圖片、聲音或專用模型。

資料來源：[Hugging Face Transformers 4.57.1 官方 Pipeline API](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines)（與本 Notebook 安裝版本相同）。不同版本的支援清單可能調整。

### 練習 3｜載入中文情緒模型

模型已經訓練完成，這裡只需要載入，不需要自己訓練。

```python
# Load an existing Chinese emotion classifier.
# 載入現成的中文情緒分類模型。
emotion_model = pipeline("text-classification", model="Johnson8187/Chinese-Emotion-Small", device=device)
```


In [ ]:
# 請在此處根據範例練習程式碼


### 練習 4｜預測一句話的情緒

修改 `text` 試試不同語氣；`result["label"]` 是分類結果，`result["score"]` 是模型分數。

```python
# Write one message to classify.
# 輸入一句想測試情緒的文字。
text = "今天拿到獎狀，我真的好開心！"

# Predict the emotion of this message.
# 預測這句話的情緒。
result = emotion_model(text)[0]

# Show the predicted label and confidence score.
# 顯示預測標籤和分數。
print("情緒標籤：", result["label"])
print("模型分數：", round(result["score"], 3))
```


In [ ]:
# 請在此處根據範例練習程式碼


## Optional｜`LABEL_0` 到 `LABEL_7` 是什麼意思？

有些模型輸出的不是中文，而是 `LABEL_0` 等數字代碼。**每個模型對代碼的定義可能不同，必須查該模型的模型卡。** 此情緒模型的對照如下，依模型作者提供的順序整理：

| 模型標籤 | 情緒 | 模型標籤 | 情緒 |
|---|---|---|---|
| LABEL_0 | 平淡語氣 | LABEL_4 | 悲傷語調 |
| LABEL_1 | 關切語調 | LABEL_5 | 疑問語調 |
| LABEL_2 | 開心語調 | LABEL_6 | 驚奇語調 |
| LABEL_3 | 憤怒語調 | LABEL_7 | 厭惡語調 |

以下程式已寫好，**直接執行即可**。


In [ ]:
# Match the model's numeric labels with Chinese emotion names.
# 將模型的數字標籤對應到中文情緒名稱。
emotion_names = {
    "LABEL_0": "平淡語氣", "LABEL_1": "關切語調",
    "LABEL_2": "開心語調", "LABEL_3": "憤怒語調",
    "LABEL_4": "悲傷語調", "LABEL_5": "疑問語調",
    "LABEL_6": "驚奇語調", "LABEL_7": "厭惡語調"
}

# Display Chinese when the model returns a LABEL_N code.
# 若模型輸出 LABEL_N，就換成對應的中文名稱。
print("中文情緒：", emotion_names.get(result["label"], result["label"]))


## Optional｜查看遮罩填空的前三名

前面只顯示第一名。以下程式已經寫好，不需要自己寫 `for` 迴圈。


In [ ]:
# Show the three candidates in a small table.
# 將三個遮罩候選答案整理成表格。

# Show only token and score.
# 僅查看候選文字與預測分數。
display(pd.DataFrame(mask_model("這個蛋糕非常[MASK]。", top_k=3))[["token_str", "score"]])


**課堂總結：**遮罩填空猜的是「文字」，情緒分類判斷的是「類別」。分數較高表示模型對該輸出較有把握，但**不保證答案一定正確**。
